In [1]:
# swap_pca_screener.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator

import statsmodels.api as sm
from sklearn.decomposition import PCA


@dataclass
class PCAScreenerConfig:
    """
    Configuration for the SwapCurvePCAScreener.

    Expected input:
    - data: pd.DataFrame
        index: datetime-like (observation dates)
        columns: tenors/maturities (e.g. "1Y", "2Y", "5Y", "10Y", "30Y")
        values: curve points (levels or changes; the screener can transform)
    """
    n_components: int = 3

    # "full_sample" or "rolling"
    window_mode: str = "full_sample"
    rolling_window: int = 252  # e.g. 1y of trading days

    # Preprocessing
    use_changes: bool = True      # if True, run PCA on changes
    change_lag: int = 1           # diff lag if use_changes=True
    standardize_by_vol: bool = True  # scale columns by vol before PCA (per column)

    # Input filtering
    min_non_nan_frac: float = 0.85   # min fraction of non-NaN observations per column
    min_window_obs: int = 60         # minimum observations required in a rolling window


class SwapCurvePCAScreener:
    """
    PCA-based screener for relative value opportunities on a swap curve.

    Key outputs:
    - Signals: residual z-scores (mispricing) per tenor/date
    - Residuals: raw residuals in original units
    - Rolling diagnostics: loadings and explained variance ratios (rolling mode)
    """

    def __init__(self, config: Optional[PCAScreenerConfig] = None) -> None:
        self.config = config or PCAScreenerConfig()
        self._check_config()

        # Populated after fit()
        self.columns: Optional[pd.Index] = None

        # Full-sample objects
        self.full_pca: Optional[PCA] = None
        self.full_scaler: Optional[pd.Series] = None  # per-column vol scaler (if used)
        self.full_results: Optional[pd.DataFrame] = None     # z-scores per tenor
        self.full_residuals: Optional[pd.DataFrame] = None   # raw residuals in original units

        # Rolling objects
        self.rolling_results: Optional[pd.DataFrame] = None  # z-scores (last obs per window)
        self.rolling_residuals: Optional[pd.DataFrame] = None  # raw residuals (last obs per window)
        self.rolling_loadings: Optional[pd.DataFrame] = None     # MultiIndex: (date, pc) x tenors
        self.rolling_explained_var_ratio: Optional[pd.DataFrame] = None  # index date, cols pc_1, pc_2, ...

    # ---------------------------------------------------------------------
    # Public API
    # ---------------------------------------------------------------------

    def fit(self, data: pd.DataFrame) -> "SwapCurvePCAScreener":
        """
        Fit PCA according to configuration and compute residual z-scores.

        After calling fit(), you can access:
        - get_signals() -> z-scores per tenor
        - get_residuals() -> raw residuals per tenor (original units)
        - screen_* methods for opportunities
        """
        clean = self._sanitize_input(data)
        X = self._preprocess(clean)

        if self.config.window_mode == "full_sample":
            self._fit_full_sample(X)
        else:
            self._fit_rolling(X)

        return self

    def get_signals(self) -> pd.DataFrame:
        """
        Return DataFrame of residual z-scores (screening signals).
        Columns: tenors. Index: dates.
        """
        if self.config.window_mode == "full_sample":
            if self.full_results is None:
                raise RuntimeError("fit() must be called before get_signals().")
            return self.full_results.copy()
        else:
            if self.rolling_results is None:
                raise RuntimeError("fit() must be called before get_signals().")
            return self.rolling_results.copy()

    def get_residuals(self) -> pd.DataFrame:
        """
        Return DataFrame of raw residuals (not z-scored), in original units.

        In rolling mode this returns residuals for the end-of-window observation dates.
        """
        if self.config.window_mode == "full_sample":
            if self.full_residuals is None:
                raise RuntimeError("fit() must be called before get_residuals().")
            return self.full_residuals.copy()
        else:
            if self.rolling_residuals is None:
                raise RuntimeError("fit() must be called before get_residuals().")
            return self.rolling_residuals.copy()

    def top_opportunities(
        self,
        as_of: Optional[pd.Timestamp] = None,
        n: int = 10,
        abs_threshold: float = 2.0,
    ) -> pd.DataFrame:
        """
        Convenience wrapper: return the top |z| tenors for a given date.

        Returns DataFrame with columns:
        - tenor
        - z_score
        """
        signals = self.get_signals()
        if signals.empty:
            raise RuntimeError("No signals available; did you call fit()?")

        if as_of is None:
            as_of = signals.index[-1]
        else:
            as_of = pd.Timestamp(as_of)

        if as_of not in signals.index:
            raise ValueError(f"No signals available for date {as_of}.")

        row = signals.loc[as_of]
        df = row.to_frame("z_score")
        df["tenor"] = df.index
        df = df.reset_index(drop=True)

        df = df.loc[df["z_score"].abs() >= abs_threshold]
        df = df.reindex(df["z_score"].abs().sort_values(ascending=False).index)
        return df.head(n).reset_index(drop=True)

    def screen_tenors(
        self,
        as_of: Optional[pd.Timestamp] = None,
        n: int = 10,
        abs_threshold: float = 1.5,
    ) -> pd.DataFrame:
        """
        Single-tenor screening (still on z-scores).
        """
        return self.top_opportunities(as_of=as_of, n=n, abs_threshold=abs_threshold)

    def screen_curve_spreads_from_residuals(
        self,
        as_of: Optional[pd.Timestamp] = None,
        tenor_order: Optional[Sequence[str]] = None,
        max_gap: Optional[int] = None,
        min_history: int = 60,
        abs_threshold: float = 1.5,
        forward_label: str = "spot",
        n: Optional[int] = 20,
    ) -> pd.DataFrame:
        """
        Build 2-leg curve spreads using raw residuals and compute a time-series z-score.

        For each pair (t1, t2) with t2 longer than t1 in tenor_order, define:
            spread_t = resid(t2) - resid(t1)
            z_asof = (spread_asof - mean(spread_hist)) / std(spread_hist)

        Interpretation:
            z < 0  -> "steepener" (curve too flat vs history)
            z > 0  -> "flattener" (curve too steep vs history)
        """
        resid = self.get_residuals()
        if resid.empty:
            raise RuntimeError("No residuals available; did you call fit()?")

        # as_of handling
        if as_of is None:
            as_of = resid.index[-1]
        else:
            as_of = pd.Timestamp(as_of)

        if as_of not in resid.index:
            raise ValueError(f"No residuals available for date {as_of}.")

        # tenor order
        if tenor_order is None:
            tenors = list(resid.columns)
        else:
            tenors = [t for t in tenor_order if t in resid.columns]

        if len(tenors) < 2:
            raise ValueError("tenor_order must contain at least two valid tenors.")

        resid_hist = resid.loc[:as_of, tenors]

        records: List[Dict[str, Any]] = []
        for i, t1 in enumerate(tenors[:-1]):
            for j in range(i + 1, len(tenors)):
                t2 = tenors[j]

                if max_gap is not None and (j - i) > max_gap:
                    continue

                s = (resid_hist[t2] - resid_hist[t1]).dropna()
                if len(s) < min_history:
                    continue

                mu = float(s.mean())
                sigma = float(s.std(ddof=1))
                if sigma == 0 or np.isnan(sigma):
                    continue

                z_asof = float((s.iloc[-1] - mu) / sigma)
                if np.isnan(z_asof) or abs(z_asof) < abs_threshold:
                    continue

                trade = "steepener" if z_asof < 0 else "flattener"

                records.append(
                    {
                        "Forward": forward_label,
                        "Tenor1": t1,
                        "Tenor2": t2,
                        "ZScore_Spread": z_asof,
                        "Trade": trade,
                        "FullTrade": f"{forward_label} {trade} [{t1}]-[{t2}]",
                        "HistoryLength": int(len(s)),
                    }
                )

        out = pd.DataFrame(records)
        if out.empty:
            return out

        out = out.reindex(out["ZScore_Spread"].abs().sort_values(ascending=False).index)
        if n is not None:
            out = out.head(n)
        return out.reset_index(drop=True)

    def screen_flies_from_residuals(
        self,
        as_of: Optional[pd.Timestamp] = None,
        tenor_order: Optional[Sequence[str]] = None,
        min_history: int = 60,
        abs_threshold: float = 1.5,
        forward_label: str = "spot",
        belly_gap_short: int = 1,
        belly_gap_long: int = 1,
        max_belly_gap: Optional[int] = None,
        n: Optional[int] = 20,
    ) -> pd.DataFrame:
        """
        Build 3-leg flies (short, belly, long) using raw residuals and compute z-score.

        Fly residual (classic symmetric fly):
            fly_t = 2*resid(belly) - resid(short) - resid(long)

        Interpretation:
            z < 0 -> "pay belly"
            z > 0 -> "rec belly"
        """
        resid = self.get_residuals()
        if resid.empty:
            raise RuntimeError("No residuals available; did you call fit()?")

        if as_of is None:
            as_of = resid.index[-1]
        else:
            as_of = pd.Timestamp(as_of)

        if as_of not in resid.index:
            raise ValueError(f"No residuals available for date {as_of}.")

        if tenor_order is None:
            tenors = list(resid.columns)
        else:
            tenors = [t for t in tenor_order if t in resid.columns]

        if len(tenors) < 3:
            raise ValueError("tenor_order must contain at least three valid tenors.")

        resid_hist = resid.loc[:as_of, tenors]

        records: List[Dict[str, Any]] = []
        m = len(tenors)

        for i in range(0, m - 2):
            for j in range(i + 1, m - 1):
                for k in range(j + 1, m):
                    # Enforce adjacency/gap constraints around belly
                    if (j - i) < belly_gap_short or (k - j) < belly_gap_long:
                        continue
                    if max_belly_gap is not None:
                        if (j - i) > max_belly_gap or (k - j) > max_belly_gap:
                            continue

                    s_t = tenors[i]
                    b_t = tenors[j]
                    l_t = tenors[k]

                    fly = (2.0 * resid_hist[b_t] - resid_hist[s_t] - resid_hist[l_t]).dropna()
                    if len(fly) < min_history:
                        continue

                    mu = float(fly.mean())
                    sigma = float(fly.std(ddof=1))
                    if sigma == 0 or np.isnan(sigma):
                        continue

                    z_asof = float((fly.iloc[-1] - mu) / sigma)
                    if np.isnan(z_asof) or abs(z_asof) < abs_threshold:
                        continue

                    trade = "pay belly" if z_asof < 0 else "rec belly"

                    records.append(
                        {
                            "Forward": forward_label,
                            "Short": s_t,
                            "Belly": b_t,
                            "Long": l_t,
                            "ZScore_Fly": z_asof,
                            "Trade": trade,
                            "FullTrade": f"{forward_label} {trade} [{s_t}]-[{b_t}]-[{l_t}]",
                            "HistoryLength": int(len(fly)),
                        }
                    )

        out = pd.DataFrame(records)
        if out.empty:
            return out

        out = out.reindex(out["ZScore_Fly"].abs().sort_values(ascending=False).index)
        if n is not None:
            out = out.head(n)
        return out.reset_index(drop=True)

    def explained_variance(self) -> Dict[str, Any]:
        """
        Return explained variance information for the full-sample PCA.
        (Not available in rolling mode.)
        """
        if self.config.window_mode != "full_sample":
            raise RuntimeError("explained_variance() is only available for full-sample PCA.")

        if self.full_pca is None:
            raise RuntimeError("Full-sample PCA not fitted. Call fit() first.")

        return {
            "n_components": int(self.full_pca.n_components_),
            "explained_variance": self.full_pca.explained_variance_.copy(),
            "explained_variance_ratio": self.full_pca.explained_variance_ratio_.copy(),
            "singular_values": self.full_pca.singular_values_.copy(),
        }

    def compute_rolling_residual_zscores(
        self,
        window: int = 60,
        min_periods: Optional[int] = None,
    ) -> pd.DataFrame:
        """
        Compute rolling z-scores of residuals per tenor.

        Uses:
            z = (resid - roll_mean) / roll_std
        """
        resid = self.get_residuals()
        if window <= 0:
            raise ValueError("window must be >= 1 for rolling z-scores.")

        if min_periods is None:
            min_periods = window

        roll_mean = resid.rolling(window=window, min_periods=min_periods).mean()
        roll_std = resid.rolling(window=window, min_periods=min_periods).std(ddof=1)

        z = (resid - roll_mean) / roll_std
        z = z.replace([np.inf, -np.inf], np.nan)
        return z

    def get_pca_fly_weights(
        self,
        short_tenor: str,
        belly_tenor: str,
        long_tenor: str,
        pcs: Sequence[int] = (0, 1),
        as_of: Optional[pd.Timestamp] = None,
        normalization: str = "belly",
    ) -> pd.Series:
        """
        Compute PCA-weighted fly weights for (short, belly, long),
        neutralizing exposure to specified principal components.

        pcs:
            indices of PCs to neutralize (e.g. (0,) for PC1, (0,1) for PC1&PC2)

        normalization:
            - "belly": set belly weight = 1, solve for short/long
            - "sum0": enforce weights sum to 0, solve least squares
        """
        tenors = [short_tenor, belly_tenor, long_tenor]
        if len(set(tenors)) != 3:
            raise ValueError("Tenors must be three distinct names.")

        # Get loadings matrix for selected PCs and tenors
        if self.config.window_mode == "full_sample":
            if self.full_pca is None:
                raise RuntimeError("Full-sample PCA not fitted. Call fit() first.")
            if self.columns is None:
                raise RuntimeError("Internal error: columns not set.")
            cols = list(self.columns)

            for t in tenors:
                if t not in cols:
                    raise ValueError(f"Tenor '{t}' not found in PCA columns.")

            comp = self.full_pca.components_  # shape: (n_components, n_features)
            L = pd.DataFrame(comp, index=[f"pc_{i+1}" for i in range(comp.shape[0])], columns=cols)
        else:
            if self.rolling_loadings is None:
                raise RuntimeError("Rolling loadings not available. Call fit() first.")
            if as_of is None:
                # last available date in rolling_loadings
                as_of = self.rolling_loadings.index.get_level_values("date").max()
            else:
                as_of = pd.Timestamp(as_of)

            # rolling_loadings: index (date, pc), columns tenors
            try:
                L = self.rolling_loadings.xs(as_of, level="date")
            except KeyError as exc:
                raise ValueError(f"Rolling loadings not available for date {as_of}.") from exc

            for t in tenors:
                if t not in L.columns:
                    raise ValueError(f"Tenor '{t}' not found in PCA columns at {as_of}.")

        pcs = list(pcs)
        if len(pcs) < 1:
            raise ValueError("pcs must contain at least one PC index.")
        if len(pcs) >= 3:
            # with 3 legs, you can neutralize at most 2 PCs unless you relax/overdetermine
            raise ValueError("For a 3-leg fly, you can neutralize at most 2 PCs.")

        # Select rows for requested PCs
        pc_rows = [f"pc_{i+1}" for i in pcs]
        missing = [r for r in pc_rows if r not in L.index]
        if missing:
            raise ValueError(f"Requested PC rows not available: {missing}")

        L_sel = L.loc[pc_rows, tenors]  # shape: (k, 3)

        if normalization == "belly":
            # Solve for w_short, w_long given w_belly=1 and neutrality constraints:
            # Ls*w_s + Lb*1 + Ll*w_l = 0  (per PC)
            A = L_sel[[short_tenor, long_tenor]].to_numpy()
            b = -L_sel[belly_tenor].to_numpy()

            sol, *_ = np.linalg.lstsq(A, b, rcond=None)
            w_short = float(sol[0])
            w_long = float(sol[1])
            w_belly = 1.0

            weights = pd.Series(
                [w_short, w_belly, w_long],
                index=tenors,
                name="pca_fly_weights",
            )
            return weights

        elif normalization == "sum0":
            # Constraints:
            # L * w = 0  (k equations)
            # 1' * w = 0 (sum constraint)
            A = np.vstack([L_sel.to_numpy(), np.ones((1, 3))])
            b = np.zeros((A.shape[0],))
            sol, *_ = np.linalg.lstsq(A, b, rcond=None)

            weights = pd.Series(sol, index=tenors, name="pca_fly_weights")
            return weights

        else:
            raise ValueError("normalization must be 'belly' or 'sum0'.")

    def run_ols_regression_with_plots(
        self,
        df: pd.DataFrame,
        x_col: str,
        y_col: str,
        start_date: Optional[pd.Timestamp] = None,
        end_date: Optional[pd.Timestamp] = None,
    ) -> Dict[str, Any]:
        """
        Run a simple OLS regression between two time series (x_col -> y_col),
        and produce:
          - scatter plot with regression line (points colored by date)
          - residual time series plot with +/- 1 and 2 std bands

        Note: This does NOT use PCA residuals; it works on the DataFrame you pass.
        """
        if df.empty:
            raise ValueError("Input DataFrame for OLS is empty.")

        # Ensure datetime-like index
        if not np.issubdtype(df.index.dtype, np.datetime64):
            try:
                df = df.copy()
                df.index = pd.to_datetime(df.index)
            except Exception as exc:
                raise TypeError("Index must be datetime-like or convertible for plotting.") from exc

        # Slice dates if provided
        if start_date is not None:
            start_date = pd.Timestamp(start_date)
            df = df.loc[df.index >= start_date]
        if end_date is not None:
            end_date = pd.Timestamp(end_date)
            df = df.loc[df.index <= end_date]

        missing_cols = [c for c in (x_col, y_col) if c not in df.columns]
        if missing_cols:
            raise ValueError(f"Columns {missing_cols} not found in DataFrame.")

        ts = df[[x_col, y_col]].dropna().copy()
        ts = ts.sort_index()
        if ts.empty:
            raise ValueError("No data left after NaN filtering and date slicing.")

        # OLS regression (with intercept)
        X = sm.add_constant(ts[x_col].values, has_constant="add")
        y = ts[y_col].values
        results = sm.OLS(y, X).fit()

        intercept = float(results.params[0])
        slope = float(results.params[1])

        fitted = intercept + slope * ts[x_col].values
        ts["fitted"] = fitted
        ts["residual"] = ts[y_col] - ts["fitted"]

        resid_mean = float(ts["residual"].mean())
        resid_std = float(ts["residual"].std(ddof=1))

        # Print statsmodels summary (as in notebook)
        print(results.summary())

        # -------------------------
        # Scatter plot + regression
        # -------------------------
        fig_scatter, ax_sc = plt.subplots(figsize=(8, 6))

        # color points by date (normalized to [0,1])
        dates_num = pd.Index(ts.index).view("int64").astype(float)
        c = (dates_num - dates_num.min()) / (dates_num.max() - dates_num.min() + 1e-12)

        sc = ax_sc.scatter(
            ts[x_col],
            ts[y_col],
            c=c,
            cmap="viridis",
            alpha=0.7,
            edgecolors="none",
        )
        cbar = plt.colorbar(sc, ax=ax_sc)
        cbar.set_label("Date")

        # Format colorbar ticks as dates
        n_ticks = 6
        ticks = np.linspace(0.0, 1.0, n_ticks)
        tick_vals = dates_num.min() + ticks * (dates_num.max() - dates_num.min())
        tick_dates = pd.to_datetime(tick_vals.astype("int64"))
        cbar.set_ticks(ticks)
        cbar.set_ticklabels([d.strftime("%Y-%m") for d in tick_dates])

        # Regression line
        x_grid = np.linspace(ts[x_col].min(), ts[x_col].max(), 100)
        y_grid = intercept + slope * x_grid
        ax_sc.plot(x_grid, y_grid, label="OLS fit")

        # Highlight last point
        last_x = float(ts[x_col].iloc[-1])
        last_y = float(ts[y_col].iloc[-1])
        ax_sc.scatter([last_x], [last_y], s=60, zorder=3)
        ax_sc.annotate("Last", (last_x, last_y), textcoords="offset points", xytext=(6, 6))

        title = (
            f"OLS: {y_col} regressed on {x_col}\n"
            f"y = {intercept:.3f} + {slope:.3f} * x | R²={results.rsquared:.2f}\n"
            f"Most recent: {ts.index[-1].date()}"
        )
        ax_sc.set_title(title)
        ax_sc.set_xlabel(x_col)
        ax_sc.set_ylabel(y_col)
        ax_sc.legend(loc="best")
        ax_sc.grid(True, alpha=0.3)
        plt.tight_layout()

        # -------------------------
        # Residuals time series plot
        # -------------------------
        fig_resid, ax_r = plt.subplots(figsize=(10, 4))
        ax_r.plot(ts.index, ts["residual"], label="Residuals", lw=1.2)

        ax_r.axhline(resid_mean, linestyle="--", linewidth=0.8)
        for k in (1, 2):
            ax_r.axhline(resid_mean + k * resid_std, linestyle="--", linewidth=0.8)
            ax_r.axhline(resid_mean - k * resid_std, linestyle="--", linewidth=0.8)

        ax_r.set_title(
            f"Residuals of {y_col} regressed on {x_col}\n"
            f"Mean={resid_mean:.3f}, Std={resid_std:.3f}"
        )
        ax_r.set_xlabel("Date")
        ax_r.set_ylabel("Residual")
        ax_r.grid(True, alpha=0.3)
        ax_r.legend(loc="best")
        plt.tight_layout()

        return {
            "results": results,
            "data": ts,
            "residuals": ts["residual"],
            "mean": resid_mean,
            "std": resid_std,
        }

    # ---------------------------------------------------------------------
    # Internal: config checks and preprocessing
    # ---------------------------------------------------------------------

    def _check_config(self) -> None:
        if self.config.n_components <= 0:
            raise ValueError("n_components must be positive.")
        if self.config.window_mode not in ("full_sample", "rolling"):
            raise ValueError("window_mode must be 'full_sample' or 'rolling'.")
        if self.config.window_mode == "rolling" and self.config.rolling_window <= 0:
            raise ValueError("rolling_window must be positive in rolling mode.")
        if self.config.change_lag <= 0:
            raise ValueError("change_lag must be positive.")
        if not (0.0 < self.config.min_non_nan_frac <= 1.0):
            raise ValueError("min_non_nan_frac must be in (0, 1].")
        if self.config.min_window_obs <= 0:
            raise ValueError("min_window_obs must be >= 1.")

    def _sanitize_input(self, data: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(data, pd.DataFrame):
            raise TypeError("data must be a pandas DataFrame.")
        if data.empty:
            raise ValueError("Input DataFrame is empty.")

        # drop duplicate timestamps (keep last)
        if data.index.has_duplicates:
            data = data.loc[~data.index.duplicated(keep="last")]

        # ensure datetime index
        if not np.issubdtype(data.index.dtype, np.datetime64):
            try:
                data = data.copy()
                data.index = pd.to_datetime(data.index)
            except Exception as exc:
                raise TypeError("Index must be datetime-like or convertible.") from exc

        data = data.sort_index()

        # drop columns with insufficient non-NaN coverage
        non_nan_frac = data.notna().mean(axis=0)
        keep_cols = non_nan_frac[non_nan_frac >= self.config.min_non_nan_frac].index
        if len(keep_cols) < 2:
            raise ValueError("Not enough tenors with sufficient data after NaN filtering.")

        data = data[keep_cols]

        # small gaps: forward/back fill (as per file style)
        data = data.ffill().bfill()

        self.columns = data.columns
        return data

    def _preprocess(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Apply transformations before PCA:
        - optionally use changes (diff) instead of levels
        - optionally scale by vol (per column)
        """
        if self.config.use_changes:
            X = data.diff(self.config.change_lag).dropna()
        else:
            X = data.copy()

        # scale by vol (per column)
        if self.config.standardize_by_vol:
            vol = X.std(axis=0, ddof=1)
            vol = vol.replace(0.0, np.nan)

            # avoid division by zero / all-NaN columns
            vol = vol.fillna(vol[vol > 0].median())
            X_scaled = X.divide(vol, axis=1)

            self.full_scaler = vol.copy()
            return X_scaled

        self.full_scaler = None
        return X

    # ---------------------------------------------------------------------
    # Internal: PCA engines
    # ---------------------------------------------------------------------

    def _fit_full_sample(self, X: pd.DataFrame) -> None:
        """
        Fit PCA on the entire sample and compute residual z-scores per tenor over time.
        """
        n_components = min(self.config.n_components, X.shape[1])
        pca = PCA(n_components=n_components)
        scores = pca.fit_transform(X.values)
        reconstructed = pca.inverse_transform(scores)

        resid_scaled = X.values - reconstructed
        resid_scaled_df = pd.DataFrame(resid_scaled, index=X.index, columns=X.columns)

        # back to original space (before vol scaling)
        if self.full_scaler is not None:
            resid_df = resid_scaled_df.multiply(self.full_scaler, axis=1)
        else:
            resid_df = resid_scaled_df

        self.full_pca = pca
        self.full_residuals = resid_df

        # z-score residuals per column over time
        mu = resid_df.mean(axis=0)
        sigma = resid_df.std(axis=0, ddof=1).replace(0.0, np.nan)
        z = (resid_df - mu) / sigma
        z = z.replace([np.inf, -np.inf], np.nan)

        self.full_results = z

    def _fit_rolling(self, X: pd.DataFrame) -> None:
        """
        For each rolling window, fit PCA and compute:
        - residual z-scores for the last observation in that window
        - raw residuals for the last observation in that window
        - store loadings and explained variance ratios for diagnostics
        """
        w = int(self.config.rolling_window)
        if X.shape[0] < self.config.min_window_obs:
            raise ValueError(
                "Not enough observations for rolling PCA. "
                f"Need at least min_window_obs={self.config.min_window_obs}."
            )

        all_dates: List[pd.Timestamp] = []
        all_z: List[pd.Series] = []
        all_resid_last: List[pd.Series] = []
        loadings_frames: List[pd.DataFrame] = []
        evr_frames: List[pd.DataFrame] = []

        for end_idx in range(w - 1, len(X)):
            start_idx = end_idx - w + 1
            window_X = X.iloc[start_idx : end_idx + 1]

            if window_X.shape[0] < self.config.min_window_obs:
                continue

            n_components = min(self.config.n_components, window_X.shape[1])
            pca = PCA(n_components=n_components)

            scores = pca.fit_transform(window_X.values)
            reconstructed = pca.inverse_transform(scores)
            resid_scaled = window_X.values - reconstructed
            resid_scaled_df = pd.DataFrame(resid_scaled, index=window_X.index, columns=window_X.columns)

            # back to original space (before vol scaling)
            if self.full_scaler is not None:
                resid_df = resid_scaled_df.multiply(self.full_scaler, axis=1)
            else:
                resid_df = resid_scaled_df

            # z-score within window, take last obs
            mu = resid_df.mean(axis=0)
            sigma = resid_df.std(axis=0, ddof=1).replace(0.0, np.nan)
            z_window = (resid_df - mu) / sigma
            z_window = z_window.replace([np.inf, -np.inf], np.nan)

            last_date = window_X.index[-1]
            all_dates.append(last_date)
            all_z.append(z_window.iloc[-1])
            all_resid_last.append(resid_df.iloc[-1])

            # loadings (components_) in scaled space are fine for neutrality usage
            comp = pd.DataFrame(
                pca.components_,
                index=[f"pc_{k+1}" for k in range(pca.components_.shape[0])],
                columns=window_X.columns,
            )
            # add date level
            comp.index.name = "pc"
            comp["date"] = last_date
            comp = comp.set_index("date", append=True).swaplevel(0, 1).sort_index()
            # Now index is (date, pc)
            comp.index.names = ["date", "pc"]
            loadings_frames.append(comp)

            evr = pd.DataFrame(
                [pca.explained_variance_ratio_],
                index=[last_date],
                columns=[f"pc_{k+1}" for k in range(len(pca.explained_variance_ratio_))],
            )
            evr_frames.append(evr)

        if not all_dates:
            raise RuntimeError("Rolling PCA produced no windows; check configuration.")

        idx = pd.DatetimeIndex(all_dates, name="date")
        self.rolling_results = pd.DataFrame(all_z, index=idx)
        self.rolling_residuals = pd.DataFrame(all_resid_last, index=idx)

        self.rolling_loadings = pd.concat(loadings_frames).sort_index()
        self.rolling_explained_var_ratio = pd.concat(evr_frames).sort_index()
